# 🌊 River Bank ERT — Interactive Forward Model

Simulate a 2D ERT cross-section across a river bank (minor bed).  
Adjust **layer depths**, **resistivities**, **electrode setup**, and **acquisition sequence** using the widgets below.

---

In [1]:
# ── Install / import dependencies ──────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, interactive, HBox, VBox, Label, HTML
from IPython.display import display, clear_output
from resipy import Project

API path =  /home/ben/micromamba/envs/geophysics-course/lib/python3.10/site-packages/resipy
ResIPy version =  3.6.6
cR2.exe found and up to date.
R3t.exe found and up to date.
cR3t.exe found and up to date.


## 1 · Electrode & Acquisition Settings

In [2]:
style  = {'description_width': '180px'}
layout = widgets.Layout(width='420px')

# ── Electrode parameters ───────────────────────────────────────────────────
w_n_elec = widgets.IntSlider(
    value=57, min=10, max=120, step=1,
    description='Number of electrodes',
    style=style, layout=layout)

w_spacing = widgets.FloatSlider(
    value=1.0, min=0.25, max=5.0, step=0.25,
    description='Electrode spacing (m)',
    style=style, layout=layout)

w_sequence = widgets.Dropdown(
    options=[
        ('Wenner-α  (sensitivity: layers)',            'wa'),
        ('Wenner-β  (sensitivity: vertical contacts)', 'wb'),
        ('Wenner-γ  (cross-borehole style)',           'wg'),
        ('Dipole-Dipole  (lateral resolution)',        'dpdp1'),
        ('Pole-Dipole  (deep targets)',                'pd'),
        ('Pole-Pole  (maximum depth)',                 'pp'),
        ('Schlumberger  (horizontal layers)',          'schlum'),
        ('Gradient  (fast acquisition)',               'grad'),
    ],
    value='dpdp1',
    description='Acquisition sequence',
    style=style, layout=layout)

w_noise = widgets.FloatSlider(
    value=3.0, min=0.0, max=15.0, step=0.5,
    description='Noise level (%)',
    style=style, layout=layout)

display(HTML('<b>🔌 Electrode & Acquisition</b>'))
display(VBox([w_n_elec, w_spacing, w_sequence, w_noise]))

HTML(value='<b>🔌 Electrode & Acquisition</b>')

## 2 · Channel Geometry

In [4]:
w_river_width = widgets.FloatSlider(
    value=10.0, min=2.0, max=30.0, step=0.5,
    description='Channel width (m)',
    style=style, layout=layout)

w_bank_height = widgets.FloatSlider(
    value=2.0, min=0.5, max=6.0, step=0.25,
    description='Bank height (m)',
    style=style, layout=layout)

w_bank_slope = widgets.FloatSlider(
    value=4.0, min=1.0, max=10.0, step=0.5,
    description='Bank slope width (m)',
    style=style, layout=layout)

display(HTML('<b>🏞️ Channel Geometry</b>'))
display(VBox([w_river_width, w_bank_height, w_bank_slope]))

HTML(value='<b>🏞️ Channel Geometry</b>')

## 3 · Layer Depths & Resistivities

Depths are measured **below the channel floor**.

In [12]:
# ── Layer definitions ──────────────────────────────────────────────────────
# Each layer: (label, depth_below_channel_floor, resistivity_ohm_m)

layers_config = [
    # (name,                      depth_default, res_default, colour hint)
    ('Gravel bed (channel fill)',   2.0,          150),
    ('Hyporheic zone',             4.0,           40),
    ('Saturated alluvium',         8.0,           80),
    ('Weathered bedrock',         13.0,          600),
    ('Competent bedrock',         20.0,         2500),
]

depth_widgets = []
res_widgets   = []
rows          = []

for name, depth_def, res_def in layers_config:
    wd = widgets.FloatSlider(
        value=depth_def, min=0.5, max=25.0, step=0.25,
        description=f'Base depth (m)',
        style=style, layout=layout)
    wr = widgets.IntSlider(
        value=res_def, min=1, max=5000, step=5,
        description=f'Resistivity (Ω·m)',
        style=style, layout=layout)
    depth_widgets.append(wd)
    res_widgets.append(wr)
    rows.append(VBox([
        HTML(f'<b>Layer: {name}</b>'),
        HBox([wd, wr])
    ]))

# Extra: unsaturated bank resistivity
w_res_bank = widgets.IntSlider(
    value=300, min=10, max=3000, step=10,
    description='Unsaturated bank (Ω·m)',
    style=style, layout=layout)

display(HTML('<b>🗂️ Subsurface Layers</b>'))
for r in rows:
    display(r)
display(VBox([w_res_bank]))

HTML(value='<b>🗂️ Subsurface Layers</b>')

HTML(value='<hr><b>🏦 Bank & Clay</b>')

## 4 · Run Simulation

In [11]:
run_button   = widgets.Button(description='▶  Run Forward Model',
                               button_style='success',
                               layout=widgets.Layout(width='220px', height='40px'))
invert_button = widgets.Button(description='⚙  Run Inversion',
                                button_style='warning',
                                layout=widgets.Layout(width='220px', height='40px'))
out = widgets.Output()

# Store project so inversion button can reuse it
state = {'k': None}

def build_model(_=None):
    with out:
        clear_output(wait=True)

        # ── Read widget values ─────────────────────────────────────────────
        n_elec      = w_n_elec.value
        spacing     = w_spacing.value
        sequence    = w_sequence.value
        noise       = w_noise.value / 100.0

        river_width = w_river_width.value
        bank_height = w_bank_height.value
        bank_slope_w= w_bank_slope.value

        layer_depths = [wd.value for wd in depth_widgets]
        layer_res    = [wr.value for wr in res_widgets]
        res_bank     = w_res_bank.value

        # ── Derived geometry ───────────────────────────────────────────────
        total_length  = n_elec * spacing + 4.0
        river_center  = total_length / 2.0
        x_left_top    = river_center - river_width/2 - bank_slope_w
        x_left_toe    = river_center - river_width/2
        x_right_toe   = river_center + river_width/2
        x_right_top   = river_center + river_width/2 + bank_slope_w
        z_bank_top    = 0.0
        z_river_bed   = -bank_height

        def topo_z(x):
            if x <= x_left_top:
                return z_bank_top
            elif x <= x_left_toe:
                t = (x - x_left_top) / (x_left_toe - x_left_top)
                return z_bank_top + t * (z_river_bed - z_bank_top)
            elif x <= x_right_toe:
                return z_river_bed
            elif x <= x_right_top:
                t = (x - x_right_toe) / (x_right_top - x_right_toe)
                return z_river_bed + t * (z_bank_top - z_river_bed)
            else:
                return z_bank_top

        def water_table_z(x):
            dist = max(0, abs(x - river_center) - river_width/2)
            return z_river_bed + dist * 0.04

        # ── Electrodes (full cross-section, continuous) ────────────────────
        x_elec = np.linspace(2.0, 2.0 + (n_elec - 1) * spacing, n_elec)
        z_elec = np.array([topo_z(x) for x in x_elec])
        elec   = np.column_stack([x_elec, z_elec])

        # ── Preview topography ─────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(12, 3))
        x_plot = np.linspace(0, total_length, 500)
        z_plot = np.array([topo_z(x) for x in x_plot])
        ax.fill_between(x_plot, z_plot, z_plot.min() - 0.5,
                         color='#c8a97e', alpha=0.4, label='Subsurface')
        ax.plot(x_plot, z_plot, 'k-', lw=2, label='Surface')
        ax.plot(x_elec, z_elec, 'rv', ms=6, label=f'Electrodes (n={n_elec}, Δ={spacing}m)')
        # Water table
        wt_z = np.array([water_table_z(x) for x in x_plot])
        ax.plot(x_plot, wt_z, 'b--', lw=1.2, alpha=0.6, label='Water table')
        ax.set_xlabel('Distance (m)')
        ax.set_ylabel('Elevation (m)')
        ax.set_title(f'Profile preview — sequence: {sequence.upper()}  |  '
                     f'{n_elec} electrodes @ {spacing} m spacing  |  '
                     f'Profile length: {total_length:.1f} m')
        ax.legend(loc='lower right', fontsize=8)
        ax.set_aspect('equal')
        plt.tight_layout()
        plt.show()

        # ── Build ResIPy project ───────────────────────────────────────────
        print('Building model...')
        k = Project(typ='R2')
        k.setElec(elec)

        # Sequence
        k.createSequence([('dpdp', 1, 8, 1, 8)]) # dipole-dipole sequence
        k.createMesh(cl=0.5, res0=300)

        # ── Layer polygons ─────────────────────────────────────────────────
        # Enforce depth ordering (each layer base must be deeper than previous)
        depths = sorted(layer_depths)

        def horiz_poly(z_top, z_bot):
            return np.array([
                [0.0,          z_top],
                [total_length, z_top],
                [total_length, z_bot],
                [0.0,          z_bot],
                [0.0,          z_top],
            ])

        # Layers are defined relative to z_river_bed (channel floor = 0 reference)
        layer_tops = [z_river_bed] + [z_river_bed - d for d in depths[:-1]]
        layer_bots = [z_river_bed - d for d in depths]

        for i, (zt, zb, res) in enumerate(zip(layer_tops, layer_bots, layer_res)):
            poly = horiz_poly(zt, zb)
            k.addRegion(poly, res0=res, iplot=False)

        # Unsaturated banks (above water table)
        left_bank_unsat = np.array([
            [0.0,        topo_z(0.0)],
            [x_left_toe, topo_z(x_left_toe)],
            [x_left_toe, water_table_z(x_left_toe)],
            [0.0,        water_table_z(0.0)],
            [0.0,        topo_z(0.0)],
        ])
        right_bank_unsat = np.array([
            [x_right_toe,  topo_z(x_right_toe)],
            [total_length, topo_z(total_length)],
            [total_length, water_table_z(total_length)],
            [x_right_toe,  water_table_z(x_right_toe)],
            [x_right_toe,  topo_z(x_right_toe)],
        ])
        k.addRegion(left_bank_unsat,  res0=res_bank, iplot=False)
        k.addRegion(right_bank_unsat, res0=res_bank, iplot=False)

        # ── Initial model plot ─────────────────────────────────────────────
        print('Plotting initial resistivity model...')
        k.mesh.show(attr='res0')
        plt.title('Initial resistivity model (before inversion)')
        plt.show()

        # ── Forward model ──────────────────────────────────────────────────
        print(f'Running forward model (noise={noise*100:.1f}%)...')
        k.forward(noise=noise, iplot=True)
        plt.show()

        state['k'] = k
        print('✅ Forward model complete. Press ⚙ Run Inversion to invert.')


def run_inversion(_=None):
    with out:
        k = state.get('k')
        if k is None:
            print('⚠️  Run the forward model first.')
            return
        print('Running inversion...')
        k.invert()
        k.showResults(index=0, electrodes=True, hor_cbar=False)
        plt.title('True resistivity model')
        plt.show()
        k.showResults(index=1, electrodes=True)
        plt.title('Inverted resistivity model')
        plt.show()
        print('✅ Inversion complete.')


run_button.on_click(build_model)
invert_button.on_click(run_inversion)

display(HBox([run_button, invert_button]))
display(out)

Output()

---
## 📋 Quick Reference — Acquisition Sequences

| Sequence | Best for | Depth | Resolution |
|---|---|---|---|
| **Wenner-α** | Horizontal layers, water table | Medium | Low lateral |
| **Wenner-β** | Vertical contacts, faults | Medium | Good lateral |
| **Dipole-Dipole** | Lateral features, gravel bars | Shallow-medium | Very high |
| **Pole-Dipole** | Deep targets, asymmetric | Deep | Good |
| **Pole-Pole** | Maximum depth | Very deep | Low |
| **Schlumberger** | 1D layered stratigraphy | Medium-deep | Low lateral |
| **Gradient** | Fast surveys, large areas | Medium | Good lateral |

## 📋 Layer Resistivity Reference

| Material | Typical Ω·m |
|---|---|
| River water | 10–50 |
| Clay / clay lens | 2–20 |
| Saturated alluvium | 30–100 |
| Hyporheic zone | 20–60 |
| Saturated gravel | 80–200 |
| Dry/unsaturated bank | 150–500 |
| Weathered bedrock | 300–1000 |
| Competent bedrock | 1000–5000 |